### Setting

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import unicodedata
import re

from pathlib import Path
BASE_DIR = Path().resolve().parent   
DATA_DIR = "G:/我的云端硬盘/NLP_project"

# from google.colab import drive
# drive.mount('/content/drive')
# DATA_DIR = Path("/content/drive/MyDrive/NLP_project")

def show_full(df):
    """
    Fully display a DataFrame without any truncation.
    Works for any df[...] slice.
    """
    import pandas as pd
    from IPython.display import display

    with pd.option_context(
        "display.max_colwidth", None,
        "display.max_rows", None,
        "display.max_columns", None,
        "display.width", None
    ):
        display(df)

In [2]:
import torch
if torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"

print(f"Using {device} as device.")

Using cpu as device.


### Load data

In [3]:
df_subchunks = pd.read_parquet(f"{DATA_DIR}/df_subchunks_raw.parquet")  

In [4]:
Financial_chunks = (
    df_subchunks.loc[df_subchunks["industry"] == "Financial Systems",
                     ["chunk_uid", "url", "date", "title", "topic", "chunk_text"]]
    .copy()
    .rename(columns={"topic": "topic_v0"})
)
Financial_chunks.head(2)

,chunk_uid,url,date,title,topic_v0,chunk_text
14,7_0_0_0_0,https://economictimes.indiatimes.com/news/new-...,2023-10-31,"""AI zindabad"": Big B shares AI-created image, ...",74,"""AI zindabad"": Big B shares AI-created image, ..."
22,15_0_0_0_0,https://www.mexc.com/kk-KZ/news/1093,2025-09-10,"""Bazaar"" surpasses ""Cathedral"", how does crypt...",3,"""Bazaar"" surpasses ""Cathedral"", how does crypt..."


### Bertopic

In [ ]:
# ! pip install bertopic

In [8]:
### customize bertopic model
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from umap import UMAP
from sklearn.cluster import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer
from bertopic.vectorizers import ClassTfidfTransformer
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

RANDOM_STATE = 27

embedding_model = SentenceTransformer("thenlper/gte-small")  # , device="cuda"

umap_model = UMAP(
    n_neighbors=50, # smaller n_neighbors will -> local structure -> more detailed topics
    n_components=20, # smaller n_components will -> more coarse topics
    min_dist=0.1, # smaller min_dist will -> more clustered紧凑 topics
    metric="cosine",
    random_state=RANDOM_STATE
)

hdbscan_model = HDBSCAN(
    min_cluster_size=100, # smaller min_cluster_size will -> more detailed topics
    min_samples=50, # smaller min_samples -> easier get into topics -> more detailed topics, but also more noise
    metric="euclidean",
    cluster_selection_method="eom"
)

vectorizer_model = CountVectorizer(
    stop_words='english',
    ngram_range=(1, 2),
    min_df=8,                 # 关键：别太高，保住 gpt/openai/llm
    max_df=0.70,              # 砍掉跨站模板词，但不至于太猛
    token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z\-]{2,}\b",
    # 你如果想加自定义 stopwords，见下方“可选增强”
)
ctfidf_model = ClassTfidfTransformer()

# All steps together
topic_model = BERTopic(
  embedding_model=embedding_model,          # Step 1 - Extract embeddings
  umap_model=umap_model,                    # Step 2 - Reduce dimensionality
  hdbscan_model=hdbscan_model,              # Step 3 - Cluster reduced embeddings
  vectorizer_model=vectorizer_model,        # Step 4 - Tokenize topics
  ctfidf_model=ctfidf_model,                # Step 5 - Extract topic words
)

In [9]:
docs = Financial_chunks["chunk_text"].tolist()
topics, probs = topic_model.fit_transform(docs)

In [12]:
Financial_topics = topic_model.get_topic_info()
Financial_chunks_topics = Financial_chunks.assign( topic=topics,  probability=probs)

In [13]:
Financial_topics.to_csv(f"{DATA_DIR}/Industry_specific/Financial_topics.csv", index=False)
Financial_chunks_topics.to_parquet(f"{DATA_DIR}/Industry_specific/Financial_chunks_topics.parquet", index=False)

### Filter AI Impact related content

In [18]:
import os
import json
import pandas as pd
from jsonschema import validate, ValidationError

import openai
from openai import OpenAI
from tenacity import (
    retry,
    wait_exponential,
    stop_after_attempt,
    retry_if_exception_type
)

DS_API_KEY = "sk-a0a8781acaa74c52acd7e22c4249185c"
client = OpenAI(
    api_key=DS_API_KEY,
    base_url="https://api.deepseek.com",
)

In [19]:
system_msg = """
You are an expert financial industry analyst and topic classification assistant.

Your task is to classify each BERTopic topic into EXACTLY ONE of the following 3 categories,
based on the combined evidence from:
1. topic keyword representation, and
2. representative documents.

### The 3 Categories

1. "AI_Financial_Impact"
Definition:
The topic is mainly about how AI, machine learning, generative AI, LLMs, algorithms,
predictive models, intelligent automation, or related AI systems are being applied in the financial industry,
or how they are changing financial workflows, decisions, services, operations, risk, compliance,
trading, underwriting, fraud detection, research, advisory, customer service, or other financial functions.

Examples include:
- AI in banking, insurance, capital markets, asset management, fintech, payments, lending
- AI improving or changing compliance, fraud detection, credit scoring, underwriting, KYC/AML
- AI for financial forecasting, portfolio analysis, trading, robo-advisory, operations, customer support
- discussions of AI adoption, implementation, use cases, workflow transformation, productivity, risk, or impact in finance

2. "AI_Related_Stock"
Definition:
The topic is mainly about stocks, share prices, valuation, market performance, analyst recommendations,
earnings, ETFs, tickers, investment commentary, or financial-market coverage of AI-related companies,
AI-themed funds, or AI-linked securities.

Examples include:
- stock performance of Nvidia, C3.ai, Palantir, Microsoft, Alphabet, etc.
- AI boom driving stock prices
- AI ETFs, AI investing themes, AI-related equity analysis
- buy/sell/hold recommendations, valuation, price targets, earnings discussions for AI-related firms

Important:
This category is about AI as an investment or stock-market subject,
NOT AI being applied inside financial-industry workflows.

3. "General_NonAI_Financial"
Definition:
The topic is mainly about general finance, banking, insurance, markets, regulation, lending,
consumer finance, macro-financial issues, accounting, real estate finance, or other financial-sector content,
but NOT mainly about AI applications/impact in finance and NOT mainly about AI-related stocks.

Examples include:
- interest rates, mortgages, bank regulation, consumer loans
- insurance operations not centered on AI
- accounting, tax, personal finance, corporate finance
- general market or financial news without AI as the core topic
- fintech or digital finance topics where AI is not the main subject

### Decision Rules

- Return EXACTLY ONE category.
- Use both Representation and Representative_Docs jointly.
- Focus on the MAIN topical center, not incidental mentions.
- A passing mention of AI is not enough for "AI_Financial_Impact".
- A passing mention of stocks is not enough for "AI_Related_Stock".
- If the topic is fundamentally about financial-industry use of AI, choose "AI_Financial_Impact".
- If the topic is fundamentally about AI-related companies/securities as investments, choose "AI_Related_Stock".
- Otherwise choose "General_NonAI_Financial".

### Boundary Clarification

Choose "AI_Financial_Impact" when:
- AI is being used BY financial institutions, financial professionals, or financial workflows
- the topic is about application, adoption, transformation, operational impact, or use case in finance

Choose "AI_Related_Stock" when:
- the topic is about AI companies or AI themes AS STOCKS / INVESTMENTS
- the core language is about shares, ticker, buy/sell, valuation, market cap, earnings, ETF, recommendation, performance

Choose "General_NonAI_Financial" when:
- the topic is financial, but AI is absent, marginal, or not the main topic
- the topic is not primarily about AI application in finance
- the topic is not primarily about AI-related securities or stock investing

### Boilerplate / Crawl Artifact Rule
Ignore non-substantive web residue such as:
- navigation/menu fragments
- privacy/cookie text
- subscription prompts
- duplicated headers/titles
- author/date/meta residue
- social sharing links
- unrelated mixed snippets
- ad blocks or UI/button fragments
- repeated site-wide template text

Do NOT use such content as topical evidence.

### Output Requirements
Return valid JSON only with the following fields:
- "category": one of the 3 category names exactly
- "relevant_keywords": a short list of the most informative keywords/phrases supporting the choice
- "reasoning": brief explanation of why the topic belongs to that category
- "reviewing": brief review of what the representative documents are mainly about and whether they support the classification

If evidence is weak or mixed, still choose the single best category conservatively.
"""

# =========================================================
# 2. User prompt template
# =========================================================
topic_classification_user_template = """
Here is one BERTopic topic from the financial domain.

Topic ID:
{Topic}

Topic Representation:
{Representation}

Representative documents:
{Representative_Docs}

Please classify this topic into exactly one of the 3 categories:
- AI_Financial_Impact
- AI_Related_Stock
- General_NonAI_Financial

Use the topic representation as the primary topic signal and use representative documents
to confirm the main semantic focus. Return JSON only.
"""

# =========================================================
# 3. JSON schema
# =========================================================
topic_classification_schema = {
    "type": "object",
    "properties": {
        "category": {
            "type": "string",
            "enum": [
                "AI_Financial_Impact",
                "AI_Related_Stock",
                "General_NonAI_Financial"
            ]
        },
        "relevant_keywords": {
            "type": "array",
            "items": {"type": "string"}
        },
        "reasoning": {
            "type": "string"
        },
        "reviewing": {
            "type": "string"
        }
    },
    "required": ["category", "relevant_keywords", "reasoning", "reviewing"],
    "additionalProperties": False
}


In [21]:
@retry(
    retry=retry_if_exception_type((
        openai.RateLimitError,
        openai.APIConnectionError,
        openai.APITimeoutError,
        openai.InternalServerError
    )),
    wait=wait_exponential(multiplier=1, min=2, max=20),
    stop=stop_after_attempt(4),
    reraise=True
)
def get_answer(client, system_msg, user_msg, model="deepseek-chat"):
    completion = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_msg},
            {"role": "user", "content": user_msg}
        ],
        response_format={"type": "json_object"},
        temperature=0
    )
    content = completion.choices[0].message.content
    return json.loads(content)

# =========================================================
# 5. Helper functions
# =========================================================
VALID_CATEGORIES = {
    "AI_Financial_Impact",
    "AI_Related_Stock",
    "General_NonAI_Financial"
}

def safe_str(x):
    if x is None:
        return ""
    return str(x)

def format_representation(x, max_chars=2500):
    text = safe_str(x).strip()
    if not text:
        return "[No topic representation provided]"
    return text[:max_chars]

def format_representative_docs(x, max_chars=12000):
    text = safe_str(x).strip()
    if not text:
        return "[No representative text provided]"
    return text[:max_chars]

def normalize_result(result):
    fallback = {
        "category": "General_NonAI_Financial",
        "relevant_keywords": [],
        "reasoning": "No clear evidence of AI-in-finance application or AI-related stock focus.",
        "reviewing": "Representative text is unclear, mixed, or not strongly informative."
    }

    if not isinstance(result, dict):
        return fallback

    category = result.get("category")
    relevant_keywords = result.get("relevant_keywords", [])
    reasoning = result.get("reasoning", "")
    reviewing = result.get("reviewing", "")

    if category not in VALID_CATEGORIES:
        return fallback

    if not isinstance(relevant_keywords, list):
        relevant_keywords = []

    if not isinstance(reasoning, str) or not reasoning.strip():
        reasoning = fallback["reasoning"]

    if not isinstance(reviewing, str) or not reviewing.strip():
        reviewing = fallback["reviewing"]

    normalized = {
        "category": category,
        "relevant_keywords": [str(k) for k in relevant_keywords],
        "reasoning": reasoning.strip(),
        "reviewing": reviewing.strip()
    }

    try:
        validate(instance=normalized, schema=topic_classification_schema)
        return normalized
    except ValidationError:
        return fallback

In [22]:
# 6. Batch run
# =========================================================
answers = []

for idx, row in Financial_topics.iterrows():
    user_msg = topic_classification_user_template.format(
        Topic=safe_str(row.get("Topic", "")),
        Representation=format_representation(row.get("Representation", ""), max_chars=2500),
        Representative_Docs=format_representative_docs(row.get("Representative_Docs", ""), max_chars=12000)
    )

    try:
        raw_result = get_answer(
            client=client,
            system_msg=system_msg,
            user_msg=user_msg,
            model="deepseek-chat"
        )
        result = normalize_result(raw_result)
        result["api_status"] = "success"

    except Exception as e:
        err_msg = str(e)

        if "Insufficient Balance" in err_msg or "402" in err_msg:
            print(f"Stopped at row {idx}: insufficient API balance.")
            break

        result = {
            "category": None,
            "relevant_keywords": None,
            "reasoning": None,
            "reviewing": None,
            "api_status": f"failed: {err_msg}"
        }
        print(f"[Row {idx}] Failed: {e}")

    answers.append(result)

answers_df = pd.DataFrame(answers)

Financial_topics_out = pd.concat(
    [Financial_topics.iloc[:len(answers)].reset_index(drop=True),
     answers_df.reset_index(drop=True)],
    axis=1
)

Financial_topics_out.head()

,Topic,Count,Name,Representation,Representative_Docs,category,relevant_keywords,reasoning,reviewing,api_status
0,-1,824,-1_intelligence robotics_nasdaq artificial_tru...,"[intelligence robotics, nasdaq artificial, tru...",[First Trust Nasdaq Artificial Intelligence an...,AI_Related_Stock,[First Trust Nasdaq Artificial Intelligence an...,The topic focuses on an AI-themed ETF (First T...,The representative documents are primarily abo...,success
1,0,5474,0_compliance_financial institutions_processes_...,"[compliance, financial institutions, processes...",[Opportunities\n\nAdvanced Portfolio Optimizat...,AI_Financial_Impact,"[compliance, financial institutions, generativ...","The topic focuses on the application of AI, sp...",The representative documents discuss how gener...,success
2,1,5444,1_presale_xrp_decentralized_ethereum,"[presale, xrp, decentralized, ethereum, tokens...",[Stay tuned to his posts if you want to stay u...,General_NonAI_Financial,"[presale, xrp, decentralized, ethereum, tokens...",The topic representation and representative do...,The representative documents discuss cryptocur...,success
3,2,4714,2_advisor_indicator_forex_automated,"[advisor, indicator, forex, automated, timefra...",[(Note_SET FILE IS ATTACHED IN COMMENT)\nThe E...,AI_Financial_Impact,"[AI, automated, forex, trading, algorithm, pre...",The topic focuses on AI-powered automated trad...,The representative documents describe AI-drive...,success
4,3,3102,3_currencies_arabic_english_stocks currencies,"[currencies, arabic, english, stocks currencie...",[GET STARTEDMENAFN10042025003732001241ID110941...,General_NonAI_Financial,"[currencies, stocks currencies, mena, world mi...",The topic representation and representative do...,The representative documents are primarily web...,success


In [35]:
category_topics = Financial_topics_out.groupby('category')['Topic'].apply(list).to_dict()
print(category_topics)

df_ai_impact = Financial_chunks_topics[Financial_chunks_topics['topic'].isin(category_topics['AI_Financial_Impact'])]
df_ai_stock = Financial_chunks_topics[Financial_chunks_topics['topic'].isin(category_topics['AI_Related_Stock'])]

print(f"\n# of df_ai_impact chunks: {len(df_ai_impact)}")
print(f"# of df_ai_stock chunks: {len(df_ai_stock)}")

{'AI_Financial_Impact': [0, 2, 10], 'AI_Related_Stock': [-1, 4, 5, 7, 8, 9, 13, 14, 15, 18, 27, 28], 'General_NonAI_Financial': [1, 3, 6, 11, 12, 16, 17, 19, 20, 21, 22, 23, 24, 25, 26, 29]}

# of df_ai_impact chunks: 10735
# of df_ai_stock chunks: 7708


In [38]:
df_ai_impact.to_parquet(f"{DATA_DIR}/Industry_specific/finance_df_ai_impact.parquet", index=False)
df_ai_stock.to_parquet(f"{DATA_DIR}/Industry_specific/finance_df_ai_stock.parquet", index=False)

### Sentiment Analysis on chunk level

#### AI Impact

In [37]:
df_ai_impact['topic'].value_counts()    

topic
0     5474
2     4714
10     547
Name: count, dtype: int64

In [ ]:
df_ai_impact_sample = pd.concat([
    df_ai_impact[df_ai_impact["topic"] == 0].sample(50, random_state=42),
    df_ai_impact[df_ai_impact["topic"] == 2].sample(42, random_state=42),
    df_ai_impact[df_ai_impact["topic"] == 10].sample(10, random_state=42),
])[["topic", "chunk_text"]].reset_index(drop=True)

#### AI Stock

### NER: Company and Technologies

#### Identify Companies

#### Idenify Technologies 

#### How Companies are impacted by Technologies

### Aggregated Sentiment Anaysis

#### Topic Aggregation

#### Company Aggregation

#### Time Aggregation